In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

# Canonical symbolic answer:
# f_c(T) = (epsilon - 2 k_B T ln g) / d

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

def normalize_expr(text):
    if text is None:
        return ""

    s = str(text).strip()

    # Remove common labels / wrappers
    s = re.sub(r"^\s*f_c\s*\(\s*t\s*\)\s*=\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^\s*f_c\s*=\s*", "", s, flags=re.IGNORECASE)
    s = s.replace("\\boxed{", "")
    s = s.replace("$", "")
    s = s.replace("{", "")
    s = s.replace("}", "")

    # Normalize LaTeX / unicode / spacing
    replacements = {
        " ": "",
        "\n": "",
        "\t": "",
        "\\left": "",
        "\\right": "",
        "\\cdot": "*",
        "\\epsilon": "epsilon",
        "\\ln": "ln",
        "−": "-",
        "–": "-",
        "—": "-",
        "k_B": "kB",
        "k_B": "kB",
        "k_Bt": "kB*T",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    # normalize T and multiplication patterns
    s = s.replace("kbt", "kB*T")
    s = s.replace("kb*t", "kB*T")
    s = s.replace("tln", "T*ln")
    s = s.replace("2kbt", "2*kB*T")
    s = s.replace("2kb*t", "2*kB*T")
    s = s.replace("2k_b*t", "2*kB*T")
    s = s.replace("2kB*Tlng", "2*kB*T*ln(g)")
    s = s.replace("2*kB*Tlng", "2*kB*T*ln(g)")
    s = s.replace("lng", "ln(g)")
    s = s.replace("ln(g)", "ln(g)")

    # Normalize denominator variants
    s = s.replace("/(1*d)", "/d")
    s = s.replace("/(d)", "/d")

    # normalize max[...] wrappers by stripping them for structural inspection
    s = s.replace("max[0,", "max(0,")
    s = s.replace("\\max", "max")

    return s.lower()

def is_correct_fc_expr(answer_text):
    s = normalize_expr(answer_text)
    if not s:
        return False, s

    # reject explicit factor-of-2 denominator versions
    wrong_patterns = [
        r".*/\(2\*d\)",
        r".*/2d",
    ]
    for pat in wrong_patterns:
        if re.search(pat, s):
            return False, s

    # accepted patterns
    accepted_patterns = [
        r"\(epsilon-2\*kb\*t\*ln\(g\)\)/d",
        r"epsilon-2\*kb\*t\*ln\(g\)/d",  # permissive
        r"epsilon/d-2\*kb\*t\*ln\(g\)/d",
        r"\(epsilon-2kb\*t\*ln\(g\)\)/d",
        r"\(epsilon-2\*kbt\*ln\(g\)\)/d",
    ]

    for pat in accepted_patterns:
        if re.fullmatch(pat, s):
            return True, s

    # broader structural fallback:
    has_epsilon = "epsilon" in s
    has_entropy_term = ("2*kb*t*ln(g)" in s) or ("2kbt*ln(g)" in s) or ("2*kb*tlng" in s)
    has_div_d = s.endswith("/d") or "/d" in s
    has_wrong_half = ("/2d" in s) or ("/(2*d)" in s) or ("2*d" in s and s.endswith(")"))

    if has_epsilon and has_entropy_term and has_div_d and not has_wrong_half:
        return True, s

    return False, s

def classify_failure_fp_0010(answer_text):
    s = normalize_expr(answer_text)

    if not s:
        return "hallucination"

    # Dominant wrong answer in uploaded results
    if "/2d" in s or "/(2*d)" in s or re.search(r"\(epsilon-2\*kb\*t\*ln\(g\)\)/(2\*d)", s):
        return "failure_to_recognize_key_aspects"

    if "max(" in s and ("/2d" in s or "/(2*d)" in s):
        return "failure_to_recognize_key_aspects"

    if "epsilon" in s and "ln" in s:
        return "misapplication_of_equation_or_model"

    return "hallucination"

# ----------------------------
# Frontier Physics Task 010
# ----------------------------
@kbench.task(
    name="FP-0010 Hairpin Polymer Critical Peeling Force",
    description="Hard statistical-physics task on a zipper-like polymer peeling transition with a symmetric spreader, testing the correct mechanical work term and critical-force condition."
)
def fp_0010_hairpin_polymer_critical_force(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A long polymer is adsorbed on a flat substrate in a folded hairpin configuration: two antiparallel arms lie side-by-side and form identical rungs at regular spacing $d$ along the substrate. It costs an energy $\epsilon$ to peel a rung, and a rung can be peeled only if the rung immediately to its right has already peeled, so peeling is contiguous from the right.

An intact rung has a unique internal state, but peeling a rung creates two dangling ends at the peel front, and each dangling end can be in one of $g$ internal states.

At the right-hand end, the two free arms are attached to a rigid, frictionless symmetric spreader whose motion is constrained to remain centered on the midline between the arms. The actuator pulls quasistatically so that the tensions in the two arms are always equal and have constant magnitude $f$.

Assume an infinitely long polymer and fixed temperature $T$.

Question: What is the critical force $f_c(T)$ above which the polymer peels indefinitely for fixed $T$?

Return JSON only in the following format:
{
  "final_answer": "<symbolic expression>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        correct, normalized_answer = is_correct_fc_expr(final_answer)

        if correct:
            passed_checks = 1
        else:
            failure_mode = classify_failure_fp_0010(final_answer)

    trace = build_trace(
        task_id="fp_0010",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0010_hairpin_polymer_critical_force.run(kbench.llm)

In [ ]:
results = fp_0010_hairpin_polymer_critical_force.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0010"]